In [ ]:
# 1) Project setup and imports
from pathlib import Path
import torch
import torch.nn as nn

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "scripts" / "train.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Project root:", PROJECT_ROOT)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# 2) Import reusable training components from scripts/train.py
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.train import (
    set_seed,
    build_dataloaders,
    AthenAIModel,
    COMMAND_VOCAB,
    TARGET_NUM_SAMPLES,
    NUM_SENSORS,
 )

set_seed(42)
print("Loaded", len(COMMAND_VOCAB), "command classes")

In [ ]:
# 3) Build dataloaders
BATCH_SIZE = 32
DATA_DIR = PROJECT_ROOT / "data" / "mixed"

dataloaders = build_dataloaders(data_dir=DATA_DIR, batch_size=BATCH_SIZE)
train_loader = dataloaders["train"]
val_loader = dataloaders["val"]
test_loader = dataloaders["test"]

print("Train samples:", len(train_loader.dataset))
print("Val samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

In [ ]:
# 4) Build model + optimizer + loss
MODE = "full"  # change to "base" if needed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AthenAIModel(mode=MODE).to(device)
trainable_params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW(trainable_params, lr=3e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

print("Mode:", MODE)
print("Device:", device)
print("Trainable params:", sum(p.numel() for p in trainable_params))
print("Frozen params:", sum(p.numel() for p in model.parameters() if not p.requires_grad))

In [ ]:
# 5) One-batch sanity check
audio, sensor, target = next(iter(train_loader))

print("Audio shape:", tuple(audio.shape), "expected [B, 32000]")
print("Sensor shape:", tuple(sensor.shape), "expected [B, 128, 8]")
print("Target shape:", tuple(target.shape))

audio = audio.to(device)
sensor = sensor.to(device)
target = target.to(device)

model.train()
if MODE == "full":
    logits = model(audio, sensor)
else:
    logits = model(audio)

loss = criterion(logits, target)
print("Logits shape:", tuple(logits.shape), "expected [B, 20]")
print("Sanity loss:", float(loss.item()))

In [ ]:
# 6) Start training from the script (recommended)
import subprocess

cmd = [
.venv/bin/python", "scripts/train.py",
    "--mode", MODE,
    "--epochs", "50",
    "--batch_size", str(BATCH_SIZE),
    "--lr", "3e-4",
]

print("Running:", " ".join(cmd))
# Uncomment to launch training from notebook:
# subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

In [ ]:
# 7) Quick usage guide
print("Run cells 1-5 for setup and sanity checks.")
print("In cell 6, uncomment subprocess.run(...) to start training.")
print("After training, run: .venv/bin/python scripts/evaluate.py --mode full")